In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

rais = pd.read_csv(
    "/content/drive/MyDrive/Data/RAIS/RAIS_SJC.csv",
    sep=";",
    encoding="utf-8-sig"
)

print("Linhas e colunas:", rais.shape)

Linhas e colunas: (47339, 24)


In [ ]:
display(rais.head())

,Bairros SP,Bairros Fortaleza,Bairros RJ,CNAE 2.0 Classe,CNAE 95 Classe,Distritos SP,Qtd Vínculos CLT,Qtd Vínculos Ativos,Qtd Vínculos Estatutários,Ind Atividade Ano,...,Município,Natureza Jurídica,Regiões Adm DF,CNAE 2.0 Subclasse,Tamanho Estabelecimento,Tipo Estab,Tipo Estab.1,UF,IBGE Subsetor,CEP Estab
0,9997,9997,9997,93131,92614,9997,3,3,0,1,...,354990,2062,9997,9313100,2,1,Cnpj,35,21,12231675
1,9997,9997,9997,47440,52442,9997,3,3,0,1,...,354990,2062,9997,4744005,2,1,Cnpj,35,16,12231010
2,9997,9997,9997,56112,55212,9997,6,6,0,1,...,354990,2062,9997,5611205,3,1,Cnpj,35,21,12240540
3,9997,9997,9997,23303,26301,9997,14,14,0,1,...,354990,2062,9997,2330302,4,1,Cnpj,35,2,12238430
4,9997,9997,9997,45307,50300,9997,5,5,0,1,...,354990,2135,9997,4530703,3,1,Cnpj,35,16,12216550


In [ ]:
display(pd.DataFrame({"Nome das colunas": rais.columns}))

,Nome das colunas
0,Bairros SP
1,Bairros Fortaleza
2,Bairros RJ
3,CNAE 2.0 Classe
4,CNAE 95 Classe
5,Distritos SP
6,Qtd Vínculos CLT
7,Qtd Vínculos Ativos
8,Qtd Vínculos Estatutários
9,Ind Atividade Ano


In [ ]:
rais = rais.drop(columns=[
    "Bairros SP",
    "Bairros Fortaleza",
    "Bairros RJ",
    "Distritos SP",
    "Regiões Adm DF"
])

print("Total de linhas e colunas:", rais.shape)

Total de linhas e colunas: (47339, 19)


In [ ]:
display(pd.DataFrame({"Colunas restantes": rais.columns}))

,Colunas restantes
0,CNAE 2.0 Classe
1,CNAE 95 Classe
2,Qtd Vínculos CLT
3,Qtd Vínculos Ativos
4,Qtd Vínculos Estatutários
5,Ind Atividade Ano
6,Ind CEI Vinculado
7,Ind Estab Participa PAT
8,Ind Rais Negativa
9,Ind Simples


In [ ]:
!pip install xlrd

In [ ]:
caminho_layout = "/content/drive/MyDrive/Data/RAIS/RAIS_vinculos_layout2020.xls"

layout = pd.ExcelFile(caminho_layout, engine="xlrd")

print(layout.sheet_names)

['RAISD - layout', 'BAIRRO_SP', 'BAIRRO FORT', 'BAIRRO_RJ', 'ocupação', 'municipio', 'subclasse 2.0', 'classe 1.0 ou 95', 'FAIXAS', 'escolaridade ou g instruçao', 'Distrito SP', 'mes adm ou desl', 'REG ADM DF']


In [ ]:
tabela_subclasse = pd.read_excel(
    caminho_layout,
    sheet_name="subclasse 2.0",
    header=None,
    engine="xlrd"
)

display(tabela_subclasse.head(10))

,0
0,CNAE 2.0 Subclas
1,0111301:Cultivo de Arroz
2,0111302:Cultivo de Milho
3,0111303:Cultivo de Trigo
4,0111399:Cultivo de Outros Cereais não Especifi...
5,0112101:Cultivo de Algodão Herbáceo
6,0112102:Cultivo de Juta
7,0112199:Cultivo de Outras Fibras de Lavoura Te...
8,0113000:Cultivo de Cana-De-Açúcar
9,0114800:Cultivo de Fumo


In [ ]:
# Separar código e descrição
depara_subclasse = tabela_subclasse.iloc[1:, 0].dropna().astype(str).str.split(
    ":",
    n=1,
    expand=True
)

# Dar nomes às duas colunas
depara_subclasse.columns = [
    "CNAE 2.0 Subclasse",
    "Descrição CNAE Subclasse"
]

# Remover espaços desnecessários
depara_subclasse["CNAE 2.0 Subclasse"] = (
    depara_subclasse["CNAE 2.0 Subclasse"].str.strip()
)

depara_subclasse["Descrição CNAE Subclasse"] = (
    depara_subclasse["Descrição CNAE Subclasse"].str.strip()
)

# Mostrar o resultado
display(depara_subclasse.head(10))

,CNAE 2.0 Subclasse,Descrição CNAE Subclasse
1,0111301,Cultivo de Arroz
2,0111302,Cultivo de Milho
3,0111303,Cultivo de Trigo
4,0111399,Cultivo de Outros Cereais não Especificados An...
5,0112101,Cultivo de Algodão Herbáceo
6,0112102,Cultivo de Juta
7,0112199,Cultivo de Outras Fibras de Lavoura Temporária...
8,0113000,Cultivo de Cana-De-Açúcar
9,0114800,Cultivo de Fumo
10,0115600,Cultivo de Soja


In [ ]:
rais["CNAE 2.0 Subclasse"] = (
    rais["CNAE 2.0 Subclasse"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
    .str.zfill(7)
)

depara_subclasse["CNAE 2.0 Subclasse"] = (
    depara_subclasse["CNAE 2.0 Subclasse"]
    .astype(str)
    .str.strip()
    .str.zfill(7)
)

In [ ]:
rais = rais.merge(
    depara_subclasse,
    on="CNAE 2.0 Subclasse",
    how="left"
)

In [ ]:
display(
    rais[
        [
            "CNAE 2.0 Subclasse",
            "Descrição CNAE Subclasse"
        ]
    ].head(10)
)

,CNAE 2.0 Subclasse,Descrição CNAE Subclasse
0,9313100,Atividades de Condicionamento Físico
1,4744005,Comércio Varejista de Materiais de Construção ...
2,5611205,NaN
3,2330302,Fabricação de Artefatos de Cimento para Uso na...
4,4530703,Comércio a Varejo de Peças e Acessórios Novos ...
5,4754701,Comércio Varejista de Móveis
6,7739099,Aluguel de Outras Máquinas e Equipamentos Come...
7,4530703,Comércio a Varejo de Peças e Acessórios Novos ...
8,4330404,Serviços de Pintura de Edifícios em Geral
9,5250803,"Agenciamento de Cargas, Exceto para O Transpor..."


In [ ]:
print(
    "Descrições não encontradas:",
    rais["Descrição CNAE Subclasse"].isna().sum()
)

Descrições não encontradas: 1102


In [ ]:
print("Total de registros:", len(rais))

print(
    "Descrições não encontradas:",
    rais["Descrição CNAE Subclasse"].isna().sum()
)

Total de registros: 47339
Descrições não encontradas: 1102


In [ ]:
codigos_pendentes = rais.loc[
    rais["Descrição CNAE Subclasse"].isna(),
    "CNAE 2.0 Subclasse"
].value_counts()

display(codigos_pendentes)

,count
CNAE 2.0 Subclasse,
6201501,463
5611204,190
4713004,80
8020001,65
9609208,63
4541206,56
6201502,34
9412099,31
7410299,25


In [ ]:
tamanhos = {
    1: "Tamanho zero",
    2: "Até 4 empregados",
    3: "5 a 9 empregados",
    4: "10 a 19 empregados",
    5: "20 a 49 empregados",
    6: "50 a 99 empregados",
    7: "100 a 249 empregados",
    8: "250 a 499 empregados",
    9: "500 a 999 empregados",
    10: "1000 ou mais empregados"
}

rais["Descrição Tamanho Estabelecimento"] = (
    rais["Tamanho Estabelecimento"].map(tamanhos)
)

display(
    rais[
        [
            "Tamanho Estabelecimento",
            "Descrição Tamanho Estabelecimento"
        ]
    ].drop_duplicates().sort_values("Tamanho Estabelecimento")
)

,Tamanho Estabelecimento,Descrição Tamanho Estabelecimento
8,1,Tamanho zero
0,2,Até 4 empregados
2,3,5 a 9 empregados
3,4,10 a 19 empregados
55,5,20 a 49 empregados
182,6,50 a 99 empregados
89,7,100 a 249 empregados
197,8,250 a 499 empregados
228,9,500 a 999 empregados
185,10,1000 ou mais empregados


In [ ]:
print(
    "Tamanhos sem descrição:",
    rais["Descrição Tamanho Estabelecimento"].isna().sum()
)

Tamanhos sem descrição: 0


In [ ]:
# Criar uma nova coluna para os setores
rais["Macrosetor"] = "Outros/Ignorado"

# Indústria
rais.loc[
    rais["IBGE Subsetor"].between(1, 13),
    "Macrosetor"
] = "Indústria"

# Serviços industriais de utilidade pública
rais.loc[
    rais["IBGE Subsetor"] == 14,
    "Macrosetor"
] = "Serviços industriais de utilidade pública"

# Construção civil
rais.loc[
    rais["IBGE Subsetor"] == 15,
    "Macrosetor"
] = "Construção civil"

# Comércio
rais.loc[
    rais["IBGE Subsetor"].isin([16, 17]),
    "Macrosetor"
] = "Comércio"

# Serviços
rais.loc[
    rais["IBGE Subsetor"].between(18, 24),
    "Macrosetor"
] = "Serviços"

# Agropecuária
rais.loc[
    rais["IBGE Subsetor"] == 25,
    "Macrosetor"
] = "Agropecuária"

print("Setores criados:")
print(rais["Macrosetor"].value_counts())

Setores criados:
Macrosetor
Serviços                                     29248
Comércio                                     12026
Construção civil                              2812
Indústria                                     2802
Agropecuária                                   361
Serviços industriais de utilidade pública       80
Outros/Ignorado                                 10
Name: count, dtype: int64


In [ ]:
display(
    rais[["IBGE Subsetor", "Macrosetor"]]
    .drop_duplicates()
    .sort_values("IBGE Subsetor")
)

,IBGE Subsetor,Macrosetor
6054,1,Indústria
3,2,Indústria
111,3,Indústria
17,4,Indústria
77,5,Indústria
224,6,Indústria
130,7,Indústria
246,8,Indústria
471,9,Indústria
116,10,Indústria


In [ ]:
print("Quantidade de registros por setor:")

print(rais["Macrosetor"].value_counts())

print("\nTotal de registros:", len(rais))

Quantidade de registros por setor:
Macrosetor
Serviços                                     29248
Comércio                                     12026
Construção civil                              2812
Indústria                                     2802
Agropecuária                                   361
Serviços industriais de utilidade pública       80
Outros/Ignorado                                 10
Name: count, dtype: int64

Total de registros: 47339


In [ ]:
display(
    rais[
        [
            "Qtd Vínculos Ativos",
            "Qtd Vínculos CLT",
            "Qtd Vínculos Estatutários"
        ]
    ].describe()
)

,Qtd Vínculos Ativos,Qtd Vínculos CLT,Qtd Vínculos Estatutários
count,47339.000000,47339.000000,47339.000000
mean,4.366738,4.364815,0.001922
std,57.353519,57.253700,0.386428
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,1.000000,1.000000,0.000000
max,8884.000000,8884.000000,84.000000


In [ ]:
print("Total de registros:", len(rais))

print(
    "Descrições CNAE não encontradas:",
    rais["Descrição CNAE Subclasse"].isna().sum()
)

Total de registros: 47339
Descrições CNAE não encontradas: 1102


In [ ]:
rais["Descrição CNAE Subclasse"] = (
    rais["Descrição CNAE Subclasse"]
    .fillna("Descrição não encontrada no layout")
)

In [ ]:
subsetores = {
    1: "Extrativa mineral",
    2: "Indústria de produtos minerais não metálicos",
    3: "Indústria metalúrgica",
    4: "Indústria mecânica",
    5: "Material elétrico e de comunicações",
    6: "Material de transporte",
    7: "Madeira e mobiliário",
    8: "Papel, papelão, editorial e gráfica",
    9: "Borracha, fumo, couros e indústrias diversas",
    10: "Indústria química",
    11: "Indústria têxtil e do vestuário",
    12: "Indústria de calçados",
    13: "Alimentos, bebidas e álcool etílico",
    14: "Serviços industriais de utilidade pública",
    15: "Construção civil",
    16: "Comércio varejista",
    17: "Comércio atacadista",
    18: "Instituições de crédito, seguros e capitalização",
    19: "Serviços técnico-profissionais e imobiliários",
    20: "Transportes e comunicações",
    21: "Alojamento, alimentação e serviços de reparação",
    22: "Serviços médicos, odontológicos e veterinários",
    23: "Ensino",
    24: "Administração pública",
    25: "Agricultura, silvicultura e pesca",
    99: "Ignorado / não classificado"
}

In [ ]:
rais["Descrição IBGE Subsetor"] = (
    pd.to_numeric(
        rais["IBGE Subsetor"],
        errors="coerce"
    ).map(subsetores)
)

display(
    rais[
        ["IBGE Subsetor", "Descrição IBGE Subsetor"]
    ].drop_duplicates().sort_values("IBGE Subsetor")
)

,IBGE Subsetor,Descrição IBGE Subsetor
6054,1,Extrativa mineral
3,2,Indústria de produtos minerais não metálicos
111,3,Indústria metalúrgica
17,4,Indústria mecânica
77,5,Material elétrico e de comunicações
224,6,Material de transporte
130,7,Madeira e mobiliário
246,8,"Papel, papelão, editorial e gráfica"
471,9,"Borracha, fumo, couros e indústrias diversas"
116,10,Indústria química


In [ ]:
print(
    "Descrições não encontradas:",
    rais["Descrição IBGE Subsetor"].isna().sum()
)

Descrições não encontradas: 0


In [ ]:
naturezas = {
    2011: "Empresa Pública",
    2038: "Sociedade de Economia Mista",
    2046: "Sociedade Anônima Aberta",
    2054: "Sociedade Anônima Fechada",
    2062: "Sociedade Empresária Limitada",
    2127: "Sociedade em Conta de Participação",
    2135: "Empresário Individual",
    2143: "Cooperativa",
    2151: "Consórcio de Sociedades",
    2178: "Estabelecimento de Sociedade Estrangeira",
    2232: "Sociedade Simples Pura",
    2240: "Sociedade Simples Limitada",
    2305: "EIRELI de Natureza Empresária",
    2313: "EIRELI de Natureza Simples",
    2321: "Sociedade Unipessoal de Advocacia",
    2348: "Código a verificar para o ano de referência",
    3034: "Serviço Notarial e Registral (Cartório)",
    3069: "Fundação Privada",
    3077: "Serviço Social Autônomo",
    3085: "Condomínio Edilício",
    3131: "Entidade Sindical",
    3220: "Organização Religiosa",
    3271: "Órgão de Direção Local de Partido Político",
    3301: "Organização Social",
    3999: "Associação Privada",
    4014: "Empresa Individual Imobiliária",
    4022: "Segurado Especial",
    4065: "Construção Civil Pessoa Física (código legado)",
    4081: "Contribuinte Individual",
    4120: "Produtor Rural (Pessoa Física)",
    9999: "Não identificada ou ignorada"
}

In [ ]:
rais["Descrição Natureza Jurídica"] = (
    pd.to_numeric(
        rais["Natureza Jurídica"],
        errors="coerce"
    ).map(naturezas)
)

display(
    rais[
        ["Natureza Jurídica", "Descrição Natureza Jurídica"]
    ].drop_duplicates().sort_values("Natureza Jurídica")
)

,Natureza Jurídica,Descrição Natureza Jurídica
442,2011,Empresa Pública
1424,2038,Sociedade de Economia Mista
6,2046,Sociedade Anônima Aberta
17,2054,Sociedade Anônima Fechada
0,2062,Sociedade Empresária Limitada
1818,2127,Sociedade em Conta de Participação
4,2135,Empresário Individual
89,2143,Cooperativa
866,2151,Consórcio de Sociedades
173,2178,Estabelecimento de Sociedade Estrangeira


In [ ]:
print(
    "Naturezas jurídicas sem descrição:",
    rais["Descrição Natureza Jurídica"].isna().sum()
)

Naturezas jurídicas sem descrição: 0


In [ ]:
print("Total de registros:", len(rais))
print("Total de colunas:", len(rais.columns))

print("\nDescrições CNAE não encontradas:")
print(
    (
        rais["Descrição CNAE Subclasse"].isna()
        | rais["Descrição CNAE Subclasse"].eq(
            "Descrição não encontrada no layout"
        )
    ).sum()
)

print("\nSubsetores sem descrição:")
print(rais["Descrição IBGE Subsetor"].isna().sum())

print("\nNaturezas jurídicas sem descrição:")
print(rais["Descrição Natureza Jurídica"].isna().sum())

print("\nTamanhos sem descrição:")
print(rais["Descrição Tamanho Estabelecimento"].isna().sum())

Total de registros: 47339
Total de colunas: 24

Descrições CNAE não encontradas:
1102

Subsetores sem descrição:
0

Naturezas jurídicas sem descrição:
0

Tamanhos sem descrição:
0


In [ ]:
display(
    rais[
        ["Natureza Jurídica", "Descrição Natureza Jurídica"]
    ]
    .drop_duplicates()
    .sort_values("Natureza Jurídica")
)

,Natureza Jurídica,Descrição Natureza Jurídica
442,2011,Empresa Pública
1424,2038,Sociedade de Economia Mista
6,2046,Sociedade Anônima Aberta
17,2054,Sociedade Anônima Fechada
0,2062,Sociedade Empresária Limitada
1818,2127,Sociedade em Conta de Participação
4,2135,Empresário Individual
89,2143,Cooperativa
866,2151,Consórcio de Sociedades
173,2178,Estabelecimento de Sociedade Estrangeira


In [ ]:
rais.to_csv(
    "/content/drive/MyDrive/Data/RAIS/RAIS_SJC_tratada.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo salvo no Google Drive!")

Arquivo salvo no Google Drive!
